# Forest Fire Detection — Complete Pipeline
**Run all cells top to bottom. Each section builds on the previous one.**

| Section | What happens |
|---|---|
| Part 1 | Install packages, set up paths, download + split dataset |
| Part 2 | Train CNN baseline (EfficientNet-B4) |
| Part 3 | Train Hybrid CNN+Transformer and ViT baseline |
| Part 4 | Evaluate all 3 models, generate all report figures |

In [ ]:
# ── STEP 0: Install packages (run once) ──────────────────────────────────────
import subprocess, sys
packages = [
    'timm', 'albumentations', 'opencv-python', 'scikit-learn',
    'matplotlib', 'seaborn', 'pandas', 'tqdm', 'Pillow', 'kaggle', 'grad-cam'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + packages)
print('All packages installed.')

In [ ]:
# ── STEP 1: Imports, paths, device ───────────────────────────────────────────
import sys, os, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import timm
from sklearn.metrics import confusion_matrix, roc_curve, auc as sk_auc

try:
    import google.colab
    PROJECT_ROOT = Path('/content/forest_fire_detection')
    print('Environment : Google Colab')
except ImportError:
    PROJECT_ROOT = Path('..').resolve()
    print('Environment : Local')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR    = PROJECT_ROOT / 'data' / 'raw'
PROCESSED  = PROJECT_ROOT / 'data' / 'processed'
CKPT_BASE  = PROJECT_ROOT / 'outputs' / 'checkpoints'
RESULT_DIR = PROJECT_ROOT / 'outputs' / 'results'
PLOTS_DIR  = PROJECT_ROOT / 'outputs' / 'plots'

for d in [RAW_DIR, PROCESSED, CKPT_BASE, RESULT_DIR, PLOTS_DIR,
          CKPT_BASE/'baseline', CKPT_BASE/'hybrid', CKPT_BASE/'vit']:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU')

from src.dataset      import get_dataloaders, FireDataset
from src.model        import CNNBaseline
from src.hybrid_model import HybridCNNTransformer
from src.train        import train_model, compute_class_weights
from src.visualize    import plot_training_curves
from src.evaluate     import evaluate_model

print('PROJECT_ROOT :', PROJECT_ROOT)
print('Setup complete.')

---
## Part 1 — Data Preparation

In [ ]:
# ── STEP 2: Download dataset and organise into fire/ and no_fire/ ─────────────
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def count_images(folder):
    if not Path(folder).exists(): return 0
    return sum(1 for f in Path(folder).rglob('*') if f.suffix.lower() in IMAGE_EXTS)

fire_dir    = RAW_DIR / 'fire'
no_fire_dir = RAW_DIR / 'no_fire'
fire_dir.mkdir(exist_ok=True)
no_fire_dir.mkdir(exist_ok=True)

n_fire = count_images(fire_dir)
n_nf   = count_images(no_fire_dir)
print(f'Current raw counts — fire: {n_fire}, no_fire: {n_nf}')

if n_fire < 100:
    print('Downloading dataset via Kaggle CLI ...')
    ret = os.system(f'kaggle datasets download -d phylake1337/fire-dataset -p "{RAW_DIR}" --unzip')
    if ret != 0:
        print()
        print('Kaggle download failed. Manual steps:')
        print('  1. Download zip from kaggle.com/datasets/phylake1337/fire-dataset')
        print('  2. Upload it here in Colab (left sidebar > folder icon)')
        print('  3. Run this in a new cell:')
        print('     import zipfile')
        print('     zipfile.ZipFile("/content/fire-dataset.zip").extractall(str(RAW_DIR))')
        print('  4. Then re-run this cell.')

    # Organise: copy fire images from wherever they landed → fire/
    if count_images(fire_dir) < 100:
        fire_imgs = []
        for f in RAW_DIR.rglob('*'):
            if f.suffix.lower() not in IMAGE_EXTS: continue
            parts = [p.lower() for p in f.parts]
            is_nf  = any('non_fire' in p or 'no_fire' in p or 'non-fire' in p for p in parts)
            in_dst = str(f).startswith(str(fire_dir)) or str(f).startswith(str(no_fire_dir))
            if not is_nf and not in_dst:
                fire_imgs.append(f)
        for f in fire_imgs:
            shutil.copy2(f, fire_dir / f.name)
        print(f'Organised {len(fire_imgs)} fire images into {fire_dir}')

    # Organise: copy no_fire images → no_fire/
    if count_images(no_fire_dir) < 100:
        nf_imgs = []
        for f in RAW_DIR.rglob('*'):
            if f.suffix.lower() not in IMAGE_EXTS: continue
            parts = [p.lower() for p in f.parts]
            is_nf  = any('non_fire' in p or 'no_fire' in p or 'non-fire' in p for p in parts)
            in_dst = str(f).startswith(str(fire_dir)) or str(f).startswith(str(no_fire_dir))
            if is_nf and not in_dst:
                nf_imgs.append(f)
        for f in nf_imgs:
            shutil.copy2(f, no_fire_dir / f.name)
        print(f'Organised {len(nf_imgs)} no_fire images into {no_fire_dir}')

print(f'Final raw counts — fire: {count_images(fire_dir)}, no_fire: {count_images(no_fire_dir)}')
assert count_images(fire_dir) >= 100, 'fire/ is empty — check download steps above'
assert count_images(no_fire_dir) >= 50,  'no_fire/ is empty — check download steps above'

In [ ]:
# ── STEP 3: Stratified 70 / 15 / 15 split ────────────────────────────────────
SEED   = 42
RATIOS = (0.70, 0.15, 0.15)
random.seed(SEED)
np.random.seed(SEED)

# Clear processed dir so re-runs are clean
if PROCESSED.exists():
    shutil.rmtree(str(PROCESSED))
PROCESSED.mkdir(parents=True)

CLASS_MAP    = {'fire': fire_dir, 'no_fire': no_fire_dir}
split_counts = {s: {} for s in ('train', 'val', 'test')}

for cls, src in CLASS_MAP.items():
    images = sorted([p for p in src.rglob('*') if p.suffix.lower() in IMAGE_EXTS])
    random.shuffle(images)
    n  = len(images)
    n1 = int(n * RATIOS[0])
    n2 = int(n * RATIOS[1])
    parts = {'train': images[:n1], 'val': images[n1:n1+n2], 'test': images[n1+n2:]}
    for split, files in parts.items():
        dest = PROCESSED / split / cls
        dest.mkdir(parents=True, exist_ok=True)
        for f in files:
            shutil.copy2(f, dest / f.name)
        split_counts[split][cls] = len(files)

print(f'{"Split":<8}  {"fire":>8}  {"no_fire":>10}  {"Total":>8}')
print('-' * 42)
for split in ('train', 'val', 'test'):
    f  = split_counts[split].get('fire', 0)
    nf = split_counts[split].get('no_fire', 0)
    print(f'{split:<8}  {f:>8}  {nf:>10}  {f+nf:>8}')
print('Split complete.')

In [ ]:
# ── STEP 4: Sample images ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, cls in enumerate(['fire', 'no_fire']):
    imgs = sorted((PROCESSED / 'train' / cls).glob('*'))[:5]
    for col, p in enumerate(imgs):
        axes[row][col].imshow(Image.open(p).convert('RGB'))
        axes[row][col].set_title(cls, fontsize=9,
            color='firebrick' if cls == 'fire' else 'steelblue')
        axes[row][col].axis('off')
plt.suptitle('Sample training images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOTS_DIR}/sample_images.png')

In [ ]:
# ── STEP 5: Class balance chart ───────────────────────────────────────────────
splits = ['train', 'val', 'test']
x, w   = np.arange(len(splits)), 0.35
fc     = [split_counts[s].get('fire',    0) for s in splits]
nfc    = [split_counts[s].get('no_fire', 0) for s in splits]

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, fc,  w, label='fire',    color='firebrick', alpha=0.85)
b2 = ax.bar(x + w/2, nfc, w, label='no_fire', color='steelblue', alpha=0.85)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(splits)
ax.set_ylabel('Images'); ax.set_title('Class balance per split')
ax.legend(); ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'class_balance.png', dpi=120, bbox_inches='tight')
plt.show()
tf, tnf = fc[0], nfc[0]
if tf > 0 and tnf > 0:
    print(f'Train imbalance ratio: {max(tf,tnf)/min(tf,tnf):.2f}x  (fire={tf}, no_fire={tnf})')

In [ ]:
# ── STEP 6: Image resolution stats ───────────────────────────────────────────
all_imgs = list((PROCESSED / 'train').rglob('*.jpg')) + \
           list((PROCESSED / 'train').rglob('*.png'))
sample   = random.sample(all_imgs, min(300, len(all_imgs)))
widths, heights = [], []
ch_sum = np.zeros(3); ch_sq = np.zeros(3); px = 0
for p in sample:
    arr = np.array(Image.open(p).convert('RGB'), dtype=np.float32) / 255.0
    h, w = arr.shape[:2]
    widths.append(w); heights.append(h)
    ch_sum += arr.reshape(-1, 3).sum(0)
    ch_sq  += (arr ** 2).reshape(-1, 3).sum(0)
    px     += h * w
mean = ch_sum / px
std  = np.sqrt(ch_sq / px - mean ** 2)
print(f'Width  — mean:{np.mean(widths):.0f}  min:{np.min(widths)}  max:{np.max(widths)}')
print(f'Height — mean:{np.mean(heights):.0f}  min:{np.min(heights)}  max:{np.max(heights)}')
print(f'Channel mean (R,G,B): {mean.round(4)}')
print(f'Channel std  (R,G,B): {std.round(4)}')
print('\nPart 1 complete.')

---
## Part 2 — CNN Baseline (EfficientNet-B4)
Phase 1: freeze backbone, train head 5 epochs. Phase 2: fine-tune all layers 20 epochs.

In [ ]:
# ── CNN: data loaders + model ─────────────────────────────────────────────────
cnn_loaders   = get_dataloaders(data_dir=str(PROCESSED), batch_size=32, img_size=224, num_workers=0)
class_weights = compute_class_weights(cnn_loaders['train'].dataset)
print(f'Train: {len(cnn_loaders["train"].dataset)}  Val: {len(cnn_loaders["val"].dataset)}  Test: {len(cnn_loaders["test"].dataset)}')
print('Class weights:', class_weights)

cnn_model = CNNBaseline(backbone='efficientnet_b4', num_classes=2, pretrained=True)
total     = sum(p.numel() for p in cnn_model.parameters())
trainable = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f'Total params: {total:,}  |  Trainable: {trainable:,}')

In [ ]:
# ── CNN Phase 1: frozen backbone, 5 epochs ───────────────────────────────────
print('=== CNN PHASE 1: frozen backbone ===')
cnn_model.freeze_backbone()
cnn_model, cnn_h1 = train_model(
    model=cnn_model, dataloaders=cnn_loaders,
    num_epochs=5, lr=1e-3, class_weights=class_weights,
    save_dir=str(CKPT_BASE / 'baseline'), device=device,
)
print('Phase 1 done.')

In [ ]:
# ── CNN Phase 2: full fine-tune, 20 epochs ───────────────────────────────────
print('=== CNN PHASE 2: full fine-tune ===')
cnn_model.unfreeze_backbone()
cnn_model, cnn_h2 = train_model(
    model=cnn_model, dataloaders=cnn_loaders,
    num_epochs=20, lr=1e-4, class_weights=class_weights,
    save_dir=str(CKPT_BASE / 'baseline'), device=device,
)
print('Phase 2 done.')

In [ ]:
# ── CNN: save history + plot ──────────────────────────────────────────────────
cnn_history = {
    'train_loss': cnn_h1['train_loss'] + cnn_h2['train_loss'],
    'val_loss'  : cnn_h1['val_loss']   + cnn_h2['val_loss'],
    'train_acc' : cnn_h1['train_acc']  + cnn_h2['train_acc'],
    'val_acc'   : cnn_h1['val_acc']    + cnn_h2['val_acc'],
}
torch.save(cnn_history, RESULT_DIR / 'baseline_history.pt')
plot_training_curves(
    history=cnn_history, title='CNN Baseline (EfficientNet-B4)',
    save_path=str(PLOTS_DIR / 'baseline_training_curves.png'),
)
best_cnn = max(cnn_history['val_acc'])
print(f'Best val accuracy: {best_cnn:.4f}')
print('CNN training complete.')

---
## Part 3 — Hybrid CNN+Transformer and ViT Baseline

In [ ]:
# ── Hybrid: data loaders + model ─────────────────────────────────────────────
# batch_size=16 because hybrid model uses more GPU memory
hybrid_loaders  = get_dataloaders(data_dir=str(PROCESSED), batch_size=16, img_size=224, num_workers=0)
h_weights       = compute_class_weights(hybrid_loaders['train'].dataset)

hybrid = HybridCNNTransformer(
    num_classes=2, backbone='efficientnet_b4', pretrained=True,
    embed_dim=512, num_heads=8, num_layers=4, dropout=0.3,
)
total = sum(p.numel() for p in hybrid.parameters())
print(f'Hybrid model — {total:,} parameters')

In [ ]:
# ── Hybrid Phase 1: CNN frozen, 8 epochs ─────────────────────────────────────
print('=== HYBRID PHASE 1: CNN frozen ===')
hybrid.freeze_backbone()
hybrid, h_hist1 = train_model(
    model=hybrid, dataloaders=hybrid_loaders,
    num_epochs=8, lr=5e-4, class_weights=h_weights,
    save_dir=str(CKPT_BASE / 'hybrid'), device=device,
)
print('Hybrid Phase 1 done.')

In [ ]:
# ── Hybrid Phase 2: full fine-tune with differential LR, 20 epochs ───────────
print('=== HYBRID PHASE 2: full fine-tune (diff LR) ===')
hybrid.unfreeze_backbone()
hybrid, h_hist2 = train_model(
    model=hybrid, dataloaders=hybrid_loaders,
    num_epochs=20, lr=1e-4, backbone_lr_multiplier=0.1,
    class_weights=h_weights,
    save_dir=str(CKPT_BASE / 'hybrid'), device=device,
)
print('Hybrid Phase 2 done.')

In [ ]:
# ── Hybrid: save history + plot ───────────────────────────────────────────────
hybrid_history = {
    'train_loss': h_hist1['train_loss'] + h_hist2['train_loss'],
    'val_loss'  : h_hist1['val_loss']   + h_hist2['val_loss'],
    'train_acc' : h_hist1['train_acc']  + h_hist2['train_acc'],
    'val_acc'   : h_hist1['val_acc']    + h_hist2['val_acc'],
}
torch.save(hybrid_history, RESULT_DIR / 'hybrid_history.pt')
plot_training_curves(
    history=hybrid_history, title='Hybrid CNN+Transformer',
    save_path=str(PLOTS_DIR / 'hybrid_training_curves.png'),
)
best_hybrid = max(hybrid_history['val_acc'])
print(f'Best val accuracy: {best_hybrid:.4f}')
print('Hybrid training complete.')

In [ ]:
# ── ViT-B/16: build ───────────────────────────────────────────────────────────
vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=2)
print(f'ViT-B/16 params: {sum(p.numel() for p in vit.parameters()):,}')

In [ ]:
# ── ViT Phase 1: head only, 5 epochs ─────────────────────────────────────────
print('=== VIT PHASE 1: head only ===')
for name, p in vit.named_parameters():
    p.requires_grad = 'head' in name
vit, v_hist1 = train_model(
    model=vit, dataloaders=hybrid_loaders,
    num_epochs=5, lr=1e-3, class_weights=h_weights,
    save_dir=str(CKPT_BASE / 'vit'), device=device,
)
print('ViT Phase 1 done.')

In [ ]:
# ── ViT Phase 2: full fine-tune, 20 epochs ───────────────────────────────────
print('=== VIT PHASE 2: full fine-tune ===')
for p in vit.parameters():
    p.requires_grad = True
vit, v_hist2 = train_model(
    model=vit, dataloaders=hybrid_loaders,
    num_epochs=20, lr=1e-5, class_weights=h_weights,
    save_dir=str(CKPT_BASE / 'vit'), device=device,
)
print('ViT Phase 2 done.')

In [ ]:
# ── ViT: save history + plot ──────────────────────────────────────────────────
vit_history = {
    'train_loss': v_hist1['train_loss'] + v_hist2['train_loss'],
    'val_loss'  : v_hist1['val_loss']   + v_hist2['val_loss'],
    'train_acc' : v_hist1['train_acc']  + v_hist2['train_acc'],
    'val_acc'   : v_hist1['val_acc']    + v_hist2['val_acc'],
}
torch.save(vit_history, RESULT_DIR / 'vit_history.pt')
plot_training_curves(
    history=vit_history, title='ViT-B/16 Baseline',
    save_path=str(PLOTS_DIR / 'vit_training_curves.png'),
)
best_vit = max(vit_history['val_acc'])
print(f'Best val accuracy: {best_vit:.4f}')
print('ViT training complete.')

---
## Part 4 — Evaluation and Visualisation
Loads the best checkpoint of all 3 models and generates every figure needed for the report.

In [ ]:
# ── Load best checkpoints ─────────────────────────────────────────────────────
def load_ckpt(model, path, device):
    state = torch.load(path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    return model.eval().to(device)

test_loaders = get_dataloaders(data_dir=str(PROCESSED), batch_size=32, img_size=224, num_workers=0)
test_loader  = test_loaders['test']
print(f'Test samples: {len(test_loader.dataset)}')

cnn_eval = load_ckpt(
    CNNBaseline(backbone='efficientnet_b4', num_classes=2, pretrained=False),
    CKPT_BASE / 'baseline' / 'best_model.pth', device
)
hybrid_eval = load_ckpt(
    HybridCNNTransformer(num_classes=2, pretrained=False),
    CKPT_BASE / 'hybrid' / 'best_model.pth', device
)
vit_eval = load_ckpt(
    timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=2),
    CKPT_BASE / 'vit' / 'best_model.pth', device
)
print('All 3 models loaded.')

In [ ]:
# ── Evaluate all 3 models ─────────────────────────────────────────────────────
print('Evaluating CNN baseline ...')
cnn_m    = evaluate_model(cnn_eval,    test_loader, device)
print('Evaluating Hybrid ...')
hybrid_m = evaluate_model(hybrid_eval, test_loader, device)
print('Evaluating ViT ...')
vit_m    = evaluate_model(vit_eval,    test_loader, device)
print('Evaluation complete.')

In [ ]:
# ── Results comparison table ──────────────────────────────────────────────────
rows = []
for name, m in [('CNN (EfficientNet-B4)', cnn_m),
                ('ViT-B/16', vit_m),
                ('Hybrid CNN+Transformer (Ours)', hybrid_m)]:
    rows.append({
        'Model'    : name,
        'Accuracy' : round(m['accuracy'],  4),
        'Precision': round(m['precision'], 4),
        'Recall'   : round(m['recall'],    4),
        'F1-Score' : round(m['f1'],        4),
        'AUC-ROC'  : round(m['auc_roc'],   4),
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(RESULT_DIR / 'comparison_table.csv', index=False)
print(f'Saved: {RESULT_DIR}/comparison_table.csv')

In [ ]:
# ── Confusion matrix (hybrid) ─────────────────────────────────────────────────
cm = confusion_matrix(hybrid_m['labels'], hybrid_m['preds'])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Fire', 'Fire'], yticklabels=['No Fire', 'Fire'], ax=ax)
ax.set_title('Confusion Matrix — Hybrid Model')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'confusion_matrix_hybrid.png', dpi=150, bbox_inches='tight')
plt.show()
tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}  FP={fp}  TN={tn}  FN={fn}')
print(f'Recall (fire detected): {tp/(tp+fn):.4f}')

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['royalblue', 'seagreen', 'tomato']
for color, (name, m) in zip(colors, [
        ('CNN Baseline', cnn_m), ('ViT-B/16', vit_m), ('Hybrid (Ours)', hybrid_m)]):
    fpr, tpr, _ = roc_curve(m['labels'], m['probs'])
    roc_auc = sk_auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name}  (AUC={roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curves — Test Set')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Model comparison bar chart ────────────────────────────────────────────────
model_names = ['CNN\nBaseline', 'ViT-B/16', 'Hybrid\n(Ours)']
metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
metric_lbls = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
all_m       = [cnn_m, vit_m, hybrid_m]

x, w = np.arange(len(model_names)), 0.15
fig, ax = plt.subplots(figsize=(11, 5))
for i, (key, lbl) in enumerate(zip(metric_keys, metric_lbls)):
    vals = [m[key] for m in all_m]
    bars = ax.bar(x + i * w, vals, w, label=lbl)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.003,
                f'{v:.3f}', ha='center', fontsize=7)
ax.set_xticks(x + w * 2); ax.set_xticklabels(model_names)
ax.set_ylim(0.5, 1.08); ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Grad-CAM ──────────────────────────────────────────────────────────────────
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    import torchvision.transforms as T

    target_layer = [hybrid_eval.cnn.blocks[-1]]
    cam = GradCAM(model=hybrid_eval, target_layers=target_layer)
    tfm = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                     T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    fire_imgs    = sorted((PROCESSED / 'test' / 'fire').glob('*'))[:4]
    no_fire_imgs = sorted((PROCESSED / 'test' / 'no_fire').glob('*'))[:4]
    samples = [(p, 'fire') for p in fire_imgs] + [(p, 'no_fire') for p in no_fire_imgs]

    fig, axes = plt.subplots(2, len(samples), figsize=(len(samples) * 3, 6))
    for col, (p, cls) in enumerate(samples):
        raw = np.array(Image.open(p).convert('RGB').resize((224, 224)))
        inp = tfm(Image.open(p).convert('RGB')).unsqueeze(0)
        tgt = [ClassifierOutputTarget(0 if cls == 'fire' else 1)]
        gs  = cam(input_tensor=inp, targets=tgt)[0]
        ov  = show_cam_on_image(raw / 255.0, gs, use_rgb=True)
        axes[0][col].imshow(raw); axes[0][col].set_title(cls, fontsize=9); axes[0][col].axis('off')
        axes[1][col].imshow(ov);  axes[1][col].set_title('CAM', fontsize=9); axes[1][col].axis('off')
    plt.suptitle('Grad-CAM — Hybrid Model', fontsize=12)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'gradcam.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Grad-CAM saved.')
except Exception as e:
    print(f'Grad-CAM skipped: {e}')

In [ ]:
# ── Transformer attention maps ────────────────────────────────────────────────
import torch.nn.functional as F
import torchvision.transforms as T
import matplotlib.cm as cm_module
import cv2

def get_attn_weights(model, inp, layer_idx=-1):
    """Hook-free attention extraction — replays the forward pass manually.

    F.multi_head_attention_forward expects seq-first inputs [S, B, D], so x_in
    is transposed before the call; the returned weights are already [B, H, S, S].
    """
    with torch.no_grad():
        feat   = model.cnn(inp)
        feat   = model.proj(feat)
        B, _, H, W = feat.shape
        tokens = feat.flatten(2).permute(0, 2, 1)
        tokens = tokens + model.pos_embed[:, :H * W, :]

        enc = model.transformer
        x   = torch.cat([enc.cls_token.expand(B, -1, -1), tokens], dim=1)   # [B, N+1, D]

        layers = list(enc.encoder.layers)
        n      = len(layers)
        target = n + layer_idx if layer_idx < 0 else layer_idx

        for i in range(target):
            x = layers[i](x)

        layer = layers[target]
        mha   = layer.self_attn
        x_in  = layer.norm1(x) if layer.norm_first else x
        x_in  = x_in.transpose(0, 1)   # [B, N+1, D] → [N+1, B, D]  (F expects seq-first)

        _, weights = F.multi_head_attention_forward(
            x_in, x_in, x_in,
            mha.embed_dim, mha.num_heads,
            mha.in_proj_weight, mha.in_proj_bias,
            mha.bias_k, mha.bias_v, mha.add_zero_attn,
            mha.dropout, mha.out_proj.weight, mha.out_proj.bias,
            training=False, need_weights=True, average_attn_weights=False,
        )
    return weights  # [B, num_heads, N+1, N+1]

tfm = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                 T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
fire_imgs = sorted((PROCESSED / 'test' / 'fire').glob('*'))[:3]

fig, axes = plt.subplots(len(fire_imgs), 3, figsize=(10, 3.5 * len(fire_imgs)))
axes = np.array(axes).reshape(len(fire_imgs), 3)

for row, p in enumerate(fire_imgs):
    raw = np.array(Image.open(p).convert('RGB').resize((224, 224)))
    inp = tfm(Image.open(p).convert('RGB')).unsqueeze(0).to(device)

    attn     = get_attn_weights(hybrid_eval, inp, layer_idx=-1)
    cls_attn = attn[0, :, 0, 1:].mean(0).cpu().numpy()   # [N]
    sz       = int(round(len(cls_attn) ** 0.5))
    attn_map = cls_attn.reshape(sz, sz)
    attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    attn_up  = cv2.resize(attn_map, (224, 224))
    overlay  = raw / 255.0 * 0.55 + cm_module.hot(attn_up)[:, :, :3] * 0.45

    axes[row][0].imshow(raw);                  axes[row][0].set_title('Original')
    axes[row][1].imshow(attn_map, cmap='hot'); axes[row][1].set_title('Attention Map')
    axes[row][2].imshow(overlay);              axes[row][2].set_title('Overlay')
    for ax in axes[row]:
        ax.axis('off')

plt.suptitle('Transformer Attention Maps (CLS token)', fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'attention_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Attention maps saved.')


In [ ]:
# ── Failure cases ─────────────────────────────────────────────────────────────
# Remove any stale pre/forward hooks left by earlier cells that crashed mid-run
for _m in hybrid_eval.modules():
    _m._forward_pre_hooks.clear()
    _m._forward_hooks.clear()

import torchvision.transforms as T

tfm = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                 T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
# Class indices match ImageFolder alphabetical sort:
# no_fire=0 ('n' < 'f'? No — 'fire' < 'no_fire', but evaluate_model/confusion matrix
# confirms fire=1 is the positive class in this checkpoint, so we mirror that here.
IDX = {'fire': 1, 'no_fire': 0}
REV = {1: 'fire', 0: 'no_fire'}

all_test = [(p, IDX[cls]) for cls in ['fire', 'no_fire']
            for p in (PROCESSED / 'test' / cls).glob('*')]
failures = []
hybrid_eval.eval()
with torch.no_grad():
    for p, true in all_test:
        t = tfm(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
        prob = torch.softmax(hybrid_eval(t), 1)[0].cpu().numpy()
        pred = int(prob.argmax())
        if pred != true:
            failures.append((p, true, pred, float(prob[pred])))
failures.sort(key=lambda x: x[3], reverse=True)
fp_cnt = sum(1 for _, t, p, _ in failures if t == 0 and p == 1)
fn_cnt = sum(1 for _, t, p, _ in failures if t == 1 and p == 0)
print(f'Total failures: {len(failures)} / {len(all_test)}  (FP={fp_cnt}, FN={fn_cnt})')
N = min(12, len(failures))
cols = 4
rows_n = -(- N // cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(14, 3.5 * rows_n))
axes = np.array(axes).flatten()
for i, (p, true, pred, conf) in enumerate(failures[:N]):
    axes[i].imshow(Image.open(p).convert('RGB').resize((224, 224)))
    axes[i].set_title(f'True:{REV[true]}\nPred:{REV[pred]} ({conf:.0%})', color='crimson', fontsize=8)
    axes[i].axis('off')
for ax in axes[N:]: ax.axis('off')
plt.suptitle('Failure Cases — Hybrid Model', fontsize=11)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'failure_cases.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Training curves for all 3 models side by side ────────────────────────────
histories = {
    'CNN Baseline' : torch.load(RESULT_DIR / 'baseline_history.pt', map_location='cpu', weights_only=True),
    'ViT-B/16'     : torch.load(RESULT_DIR / 'vit_history.pt',      map_location='cpu', weights_only=True),
    'Hybrid (Ours)': torch.load(RESULT_DIR / 'hybrid_history.pt',   map_location='cpu', weights_only=True),
}
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (name, h) in zip(axes, histories.items()):
    ax.plot(h['train_acc'], label='Train')
    ax.plot(h['val_acc'],   label='Val', linestyle='--')
    best = max(h['val_acc'])
    ax.set_title(f'{name}\nBest val: {best:.3f}', fontsize=10)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Training Accuracy Curves', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'all_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('All outputs saved to outputs/')

---
## Done!

| Figure | File |
|---|---|
| Confusion matrix | `outputs/plots/confusion_matrix_hybrid.png` |
| ROC curves | `outputs/plots/roc_curves.png` |
| Bar chart | `outputs/plots/model_comparison_bar.png` |
| Grad-CAM | `outputs/plots/gradcam.png` |
| Attention maps | `outputs/plots/attention_maps.png` |
| Failure cases | `outputs/plots/failure_cases.png` |
| Training curves | `outputs/plots/all_training_curves.png` |
| Results CSV | `outputs/results/comparison_table.csv` |